# 04. 프리빌트 미들웨어 (Prebuilt Middleware)

> 직접 만들기 전에 **이미 있는 12+ 개 프리빌트 미들웨어**부터 알아두세요. Summarization · PII · CallLimit · Fallback 등 자주 쓰는 도구를 한 곳에 모았어요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. LangChain V1이 제공하는 12개+ 프리빌트 미들웨어의 역할과 용도를 구별할 수 있어요
2. `SummarizationMiddleware`의 트리거 조건(tokens/messages/fraction)을 상황에 맞게 설정할 수 있어요
3. `PIIMiddleware`로 이메일, 신용카드, IP 등 4가지 PII 유형을 4가지 전략(redact/mask/hash/block)으로 처리할 수 있어요
4. `ModelFallbackMiddleware`, `ToolRetryMiddleware`, `ModelCallLimitMiddleware` 등 복원력(Resilience) 미들웨어를 조합하여 안정적인 에이전트를 만들 수 있어요
5. `LLMToolSelectorMiddleware`, `TodoListMiddleware` 등 에이전트 성능 향상 미들웨어를 적용할 수 있어요

## 사전 지식

- `03-Context-Engineering.ipynb`: Model/Tool/Life-cycle Context + Runtime/State/Store 데이터 소스
- `01-Middleware-Basics.ipynb`: `AgentMiddleware` 클래스와 훅 종류 (`before_model`, `wrap_tool_call` 등)

## 프리빌트 미들웨어란?

직접 구현하지 않아도 LangChain V1이 즉시 사용 가능한 미들웨어를 12가지 이상 제공해요. 이 미들웨어들은 실무에서 자주 필요한 패턴을 정교하게 구현해 놓은 것이에요.

> **왜 프리빌트 미들웨어를 사용할까요?** 직접 구현하면 수십 줄의 코드가 필요한 기능을 한 줄로 추가할 수 있어요. 마치 스마트폰에 앱을 설치하는 것처럼, `middleware=[PIIMiddleware("email", strategy="redact")]` 한 줄이면 이메일 보호 기능이 추가돼요. 직접 정규식을 짜고 테스트하는 수고를 덜 수 있어요.

```mermaid
flowchart TD
    subgraph Memory["대화 관리"]
        SUM["SummarizationMiddleware<br/>토큰/메시지/비율 기반 요약"]
        CTX["ContextEditingMiddleware<br/>도구 사용 기록 정리"]
    end

    subgraph Privacy["보안 / 프라이버시"]
        PII["PIIMiddleware<br/>redact / mask / hash / block"]
    end

    subgraph Control["흐름 제어"]
        HITL["HumanInTheLoopMiddleware<br/>approve / edit / reject"]
        MCL["ModelCallLimitMiddleware<br/>thread / run 횟수 제한"]
        TCL["ToolCallLimitMiddleware<br/>전역 / 도구별 제한"]
    end

    subgraph Resilience["복원력"]
        MFB["ModelFallbackMiddleware<br/>캐스케이딩 폴백"]
        TRT["ToolRetryMiddleware<br/>지수 백오프 재시도"]
        MRT["ModelRetryMiddleware<br/>모델 재시도"]
    end

    subgraph Performance["성능 향상"]
        LTS["LLMToolSelectorMiddleware<br/>필요한 도구만 선택"]
        TDL["TodoListMiddleware<br/>write_todos 도구 주입"]
        LTE["LLMToolEmulatorMiddleware<br/>실행 없이 도구 응답 생성"]
    end

    classDef memory fill:#d4edda,stroke:#28a745,color:#155724
    classDef privacy fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef control fill:#cce5ff,stroke:#007bff,color:#004085
    classDef resilience fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef performance fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class SUM,CTX memory
    class PII privacy
    class HITL,MCL,TCL control
    class MFB,TRT,MRT resilience
    class LTS,TDL,LTE performance
```

### 프리빌트 미들웨어 전체 목록

| 카테고리 | 미들웨어 | 핵심 용도 |
|---------|---------|----------|
| **대화 관리** | `SummarizationMiddleware` | 토큰 한도 초과 전 자동 요약 |
| **대화 관리** | `ContextEditingMiddleware` | 도구 사용 기록(ToolUse) 제거 |
| **보안** | `PIIMiddleware` | 개인식별정보 감지 및 처리 |
| **흐름 제어** | `HumanInTheLoopMiddleware` | 조건부 사람 개입 (approve/edit/reject) |
| **흐름 제어** | `ModelCallLimitMiddleware` | 모델 호출 횟수 상한선 |
| **흐름 제어** | `ToolCallLimitMiddleware` | 도구 호출 횟수 상한선 |
| **복원력** | `ModelFallbackMiddleware` | 모델 장애 시 순서대로 다음 모델 시도 |
| **복원력** | `ToolRetryMiddleware` | 도구 실패 시 지수 백오프 재시도 |
| **복원력** | `ModelRetryMiddleware` | 모델 응답 실패 시 재시도 |
| **성능 향상** | `LLMToolSelectorMiddleware` | LLM이 필요한 도구만 골라서 제공 |
| **성능 향상** | `TodoListMiddleware` | `write_todos` 도구 자동 주입 |
| **성능 향상** | `LLMToolEmulatorMiddleware` | 실제 실행 없이 도구 응답 에뮬레이션 |

### 프로덕션 미들웨어 배치 순서 원칙

> 🔑 **핵심 개념**: 프리빌트 미들웨어를 조합할 때는 **빠르고 저렴한 것을 앞에, 느리고 비싼 것을 뒤에** 배치해요. 앞쪽에서 차단된 요청은 뒤쪽 미들웨어를 거치지 않아 비용을 절약해요.

| 순서 | 카테고리 | 이유 |
|------|---------|------|
| 1 | **보안** (PII) | 모델에 민감 정보가 전달되기 전에 제거 |
| 2 | **흐름 제어** (Limit) | 과도한 호출을 사전에 차단 |
| 3 | **복원력** (Retry, Fallback) | 실패 시 자동 복구 |
| 4 | **성능** (ToolSelector) | 도구 선택 최적화 |
| 5 | **대화 관리** (Summary) | 컨텍스트 압축은 마지막에 |

## 환경 설정

In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키 등을 읽어와요)
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# LangSmith 추적 설정 (미들웨어 동작을 시각적으로 확인해요)
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Prebuilt-Middleware"

# LangSmith 추적이 활성화되었어요.
# print(f"프로젝트: {os.environ['LANGCHAIN_PROJECT']}")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델과 공통 도구 설정
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### create_agent 에이전트 그래프 구조

`create_agent()`로 생성한 에이전트는 모두 동일한 기본 그래프 구조를 가져요: `START → agent → tools → agent → ... → END`. 미들웨어는 이 그래프의 agent 노드 **내부**에서 동작해요.

## 1. 대화 관리: SummarizationMiddleware

대화가 길어지면 토큰 비용이 증가하고 최종적으로는 모델의 컨텍스트 창 한도에 도달해요. `SummarizationMiddleware`는 이 문제를 자동으로 해결해요.

> 🔁 **복습 연결**: `01-Middleware-Basics.ipynb` §2에서 `SummarizationMiddleware`의 기본형을 잠깐 봤어요. 여기서는 `trigger`/`keep` 옵션을 깊게 다룹니다.

### 트리거(trigger) 옵션

| 트리거 형식 | 예시 | 설명 |
|------------|------|------|
| `("tokens", N)` | `("tokens", 4000)` | 메시지 총 토큰이 N 초과 시 요약 |
| `("messages", N)` | `("messages", 20)` | 메시지 수가 N 초과 시 요약 |
| `("fraction", F)` | `("fraction", 0.7)` | 모델 컨텍스트 창의 F 비율 초과 시 요약 |

여러 트리거를 리스트로 전달하면 **OR 조건**으로 작동해요 — 어느 하나라도 만족하면 요약이 실행돼요.

### keep 옵션

| keep 형식 | 예시 | 설명 |
|----------|------|------|
| `("messages", N)` | `("messages", 10)` | 요약 후 최근 N개 메시지 유지 |
| `("tokens", N)` | `("tokens", 2000)` | 요약 후 최근 N 토큰에 해당하는 메시지 유지 |

> 💡 **실무 팁**: `trigger=("fraction", 0.7)`을 사용하면 모델마다 다른 컨텍스트 창 크기를 자동으로 처리해요. `gpt-4o-mini`(128K)와 `gpt-4o`(128K) 모두 동일한 설정으로 작동해요. 고정된 토큰 수 대신 비율 기반 트리거가 더 이식성이 높아요.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트 실행
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 보안: PIIMiddleware

개인식별정보(PII: Personally Identifiable Information)가 모델에 전달되거나 로그에 기록되는 것을 방지해요. 금융, 의료, 고객 서비스 분야에서는 필수 미들웨어예요.

> 🔁 **복습 연결**: `01` §2에서 `PIIMiddleware`로 맛본 개인정보 보호를, 여기서는 5개 내장 유형 × 4개 전략 옵션으로 확장합니다. 가드레일 계층으로의 배치(Defense in Depth)는 `05-Guardrails.ipynb`에서 다뤄요.

### 지원하는 PII 유형

| PII 유형 | 감지 대상 | 예시 |
|---------|---------|------|
| `"email"` | 이메일 주소 | `user@example.com` |
| `"credit_card"` | 신용카드 번호 | `1234-5678-9012-3456` |
| `"ip"` | IP 주소 | `192.168.1.100` |
| `"mac_address"` | MAC 주소 | `00:1A:2B:3C:4D:5E` |
| `"url"` | URL 주소 | `https://private.example.com` |
| `str` (정규식) | 커스텀 패턴 | `r"010-\d{4}-\d{4}"` |

### 처리 전략 (strategy)

| 전략 | 결과 예시 | 설명 |
|-----|---------|------|
| `"redact"` | `[EMAIL REDACTED]` | 완전히 제거하고 태그로 대체 |
| `"mask"` | `jo***@example.com` | 일부만 표시, 나머지 마스킹 |
| `"hash"` | `[EMAIL:a3f9b2...]` | 해시값으로 대체 (익명화) |
| `"block"` | (예외 발생) | 요청 자체를 차단 |

> 💡 **핵심 정리**: `apply_to_input`은 사용자 입력을, `apply_to_output`은 모델 응답을, `apply_to_tool_results`는 도구 실행 결과를 처리해요. 기본값은 `apply_to_input=True`, `apply_to_output=False`, `apply_to_tool_results=False`예요. 즉 **기본은 입력만 필터링**하므로, 모델 응답이나 도구 결과의 PII도 막으려면 `apply_to_output=True`를 명시해야 해요. (출처: https://docs.langchain.com/oss/python/langchain/middleware/built-in.md )

> ⚠️ **자주 하는 실수**: `strategy="block"`은 PII가 감지되면 **요청 자체를 차단**해요. 챗봇처럼 사용자 경험이 중요한 곳에서는 `"redact"` 또는 `"mask"`를 사용하는 것이 더 자연스러워요.

In [ ]:
from langchain.agents.middleware import PIIMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 흐름 제어: ModelCallLimitMiddleware, ToolCallLimitMiddleware

에이전트가 무한 루프에 빠지거나 의도치 않게 많은 API 호출을 하는 것을 방지해요. 개발 중 디버깅이나 비용 관리에 유용해요.

> 🔁 **복습 연결**: `01` §2에서 호출 제한 미들웨어를 소개했어요. 여기서는 `thread_limit`/`run_limit`/`exit_behavior` 옵션 차이에 집중합니다.

### thread_limit vs run_limit

| 파라미터 | 범위 | 예시 |
|---------|------|------|
| `thread_limit` | 전체 대화 스레드 누적 | 체크포인터와 함께 사용 |
| `run_limit` | 단일 `.invoke()` 호출 내 | 루프 방지, 비용 제한 |

### exit_behavior 옵션

| 값 | 동작 |
|----|------|
| `"end"` | 현재 응답으로 종료 (기본값) |
| `"error"` | `ModelCallLimitError` 예외 발생 |

> ⚠️ **자주 하는 실수**: `run_limit`을 너무 낮게 설정하면 도구 호출이 많은 작업이 중간에 잘려요. 도구 하나당 최소 2회의 모델 호출(도구 선택 + 결과 해석)이 필요하다는 것을 기억해요.

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 복원력: ModelFallbackMiddleware, ToolRetryMiddleware

네트워크 오류, API 서버 장애, 일시적인 오류에도 에이전트가 계속 동작하도록 해요.

> 🔁 **복습 연결**: `01` §2에서 `ModelFallbackMiddleware`/`ToolRetryMiddleware`를 맛봤다면, 여기서는 폴백 체인과 지수 백오프 파라미터를 깊게 봅니다.

### ModelFallbackMiddleware

기본 모델이 실패할 때 등록된 순서대로 다음 모델을 시도해요. 캐스케이딩(cascading) 폴백이라고 불러요.

```
gpt-4o-mini (기본) → 실패 → gpt-4o → 실패 → claude-haiku
```

### ToolRetryMiddleware

도구 호출이 실패할 때 지수 백오프(exponential backoff)로 재시도해요.

| 파라미터 | 기본값 | 설명 |
|---------|-------|------|
| `max_retries` | 3 | 최대 재시도 횟수 |
| `initial_delay` | 1.0 | 첫 재시도 전 대기 시간 (초) |
| `backoff_factor` | 2.0 | 대기 시간 배수 (1초 → 2초 → 4초) |
| `max_delay` | 30.0 | 최대 대기 시간 (초) |
| `jitter` | True | 무작위 지터 추가 |

> 💡 **실무 팁**: `jitter=True`는 동시에 많은 요청이 실패했을 때 모두 같은 시간에 재시도하여 서버를 다시 과부하시키는 **Thundering Herd** 문제를 방지해요. 프로덕션에서는 항상 켜두는 것을 권장해요.

In [ ]:
from langchain.agents.middleware import ModelFallbackMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델 실패 시 순서대로 다음 모델 시도
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.agents.middleware import ToolRetryMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 성능 향상: LLMToolSelectorMiddleware

도구가 많을수록 모델의 컨텍스트를 소비하고, LLM이 올바른 도구를 선택하기 어려워져요. `LLMToolSelectorMiddleware`는 현재 대화에 실제로 필요한 도구만 선별하여 모델에 제공해요.

이 미들웨어는 내부적으로 **별도의 LLM 호출**로 도구를 선별해요:
1. 현재 대화 내용을 분석해요
2. 전체 도구 목록 중 필요한 것만 골라요
3. 선별된 도구만 메인 모델에 전달해요

> 💡 **핵심 정리**: 도구가 5개 이하일 때는 오히려 불필요한 LLM 호출이 추가되어 비효율적이에요. **10개 이상의 도구**를 가진 에이전트에서 효과가 두드러져요. 도구 수와 `max_tools` 설정에 따른 성능 트레이드오프를 이해하는 것이 중요해요.

In [ ]:
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.tools import tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 성능 향상: TodoListMiddleware

`TodoListMiddleware`는 에이전트에 `write_todos` 도구를 자동으로 주입해요. 이 도구를 통해 에이전트가 복잡한 작업을 처리할 때 할 일 목록을 관리하고, 진행 상황을 추적할 수 있어요.

에이전트가 `write_todos` 도구를 호출하면:
1. State의 `todos` 필드가 업데이트돼요
2. 사용자에게 진행 상황이 명확하게 보여요
3. 복잡한 멀티스텝 작업의 완료율을 추적할 수 있어요

> 🔑 **핵심 개념**: `TodoListMiddleware`는 도구를 **주입(inject)**하는 미들웨어예요. `create_agent`의 `tools` 파라미터에 직접 추가하지 않아도, 미들웨어가 에이전트 실행 중 자동으로 도구를 추가해요.

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. 대화 관리: ContextEditingMiddleware

`ContextEditingMiddleware`는 대화 히스토리에서 특정 유형의 메시지를 제거해요. 가장 일반적인 용도는 `ClearToolUsesEdit`로 도구 사용 기록(ToolMessage)을 제거하는 것이에요.

**도구 사용 기록을 제거하는 이유:**
- 반복적인 도구 호출 결과가 누적되면 컨텍스트를 낭비해요
- 특정 시점 이후로는 과거 도구 결과가 불필요해요
- 요약 전 불필요한 도구 기록을 제거하여 요약 품질을 높여요

> 💡 **실무 팁**: `ContextEditingMiddleware`와 `SummarizationMiddleware`를 함께 쓸 때는 `ContextEditingMiddleware`를 **먼저** 배치하세요. 도구 기록을 제거한 후 요약하면 요약 내용이 더 정제되어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 이 셀의 핵심 구현 코드
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. 종합 예제: 프로덕션 레벨 에이전트

실무에서는 여러 프리빌트 미들웨어를 조합하여 안정적이고 안전한 에이전트를 만들어요. 이 예제는 다음 미들웨어를 함께 적용해요:

| 순서 | 미들웨어 | 역할 |
|-----|---------|------|
| 1 | `PIIMiddleware` | 사용자 입력의 개인정보 보호 (가장 먼저!) |
| 2 | `ModelCallLimitMiddleware` | 비용 과다 방지 |
| 3 | `ToolRetryMiddleware` | 외부 API 실패 복원 |
| 4 | `ModelFallbackMiddleware` | 모델 장애 대비 |
| 5 | `LLMToolSelectorMiddleware` | 도구 선택 최적화 |
| 6 | `ContextEditingMiddleware` | 도구 기록 정리 |
| 7 | `SummarizationMiddleware` | 컨텍스트 압축 |

> 💡 **핵심 정리**: 미들웨어 순서는 에이전트의 **동작 방식**을 결정해요. PII 보호는 항상 앞에, 컨텍스트 관리(요약)는 항상 마지막에 배치하는 것이 일반적인 원칙이에요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 9. 실습: 맞춤형 미들웨어 스택 설계

아래 시나리오에 맞는 미들웨어 스택을 설계하고 구현해봐요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 의료 상담 챗봇을 위한 미들웨어 스택 설계
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 10. 실습 파일 안내: cost_tracker.py, language_router.py

이 노트북과 함께 제공되는 두 개의 실습 파일에서 프리빌트 미들웨어를 활용한 실용적인 구현 예시를 확인할 수 있어요.

### `middleware/cost_tracker.py`

`ToolCallLimitMiddleware`와 커스텀 클래스를 조합하여 **토큰 사용량과 API 호출 비용을 실시간으로 추적**해요.

```python
from middleware.cost_tracker import CostTrackerMiddleware

tracker = CostTrackerMiddleware(budget_usd=1.0)  # $1.00 예산 설정
agent = create_agent(model=model, tools=[...], middleware=[tracker])
# 실행 후:
tracker.print_cost_report()  # 비용 보고서 출력
```

### `middleware/language_router.py`

`@dynamic_prompt`를 활용하여 **입력 언어를 자동 감지하고 언어별 시스템 프롬프트로 전환**해요.

```python
from middleware.language_router import LanguageRouterMiddleware

agent = create_agent(
    model=model,
    tools=[...],
    middleware=[LanguageRouterMiddleware()],
)
# 한국어 입력 → 한국어 시스템 프롬프트
# 영어 입력 → 영어 시스템 프롬프트
```

In [ ]:
from middleware.cost_tracker import CostTrackerMiddleware
from middleware.language_router import LanguageRouterMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from middleware.language_router import LanguageRouterMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **SummarizationMiddleware**: `trigger` 옵션으로 tokens/messages/fraction 기반 요약을 트리거하고, 여러 조건을 리스트로 전달하면 OR 논리로 작동해요
- **PIIMiddleware**: email, credit_card, ip, mac_address, url과 커스텀 정규식 패턴을 지원하며, redact/mask/hash/block 4가지 전략으로 처리해요
- **ModelCallLimitMiddleware / ToolCallLimitMiddleware**: `run_limit`으로 단일 실행 내 호출 횟수를 제한하고, `exit_behavior`로 종료 방식을 선택해요
- **ModelFallbackMiddleware**: 기본 모델 실패 시 등록된 순서대로 다음 모델을 캐스케이딩 시도해요
- **ToolRetryMiddleware**: `backoff_factor`로 지수 백오프를, `jitter=True`로 Thundering Herd 방지를 설정해요
- **LLMToolSelectorMiddleware**: `max_tools`와 `always_include`로 각 호출마다 필요한 도구만 선별하여 컨텍스트를 효율화해요
- **TodoListMiddleware**: `write_todos` 도구를 자동 주입하여 복잡한 멀티스텝 작업의 진행 상황을 추적해요
- **ContextEditingMiddleware**: `ClearToolUsesEdit`로 도구 사용 기록이 누적되는 것을 방지해요
- **미들웨어 순서 원칙**: PII 보호 → 호출 제한 → 복원력 → 성능 최적화 → 컨텍스트 관리 순으로 배치해요

## 다음 노트북 예고

다음 `05-Guardrails.ipynb`에서는 **가드레일(Guardrails)**을 배워요. 오늘 배운 `PIIMiddleware`를 넘어 더 강력한 PII 보호(Redact/Mask/Hash/Block)와 커스텀 안전 필터를 직접 구현해볼게요. 특정 주제나 유해 콘텐츠를 완전히 차단하는 프로덕션 수준의 가드레일 시스템을 만들어봐요.